In [119]:
!pip install segmentation-models-pytorch

In [1]:
import os 
import pandas as pd 
from torchvision.io import read_image 
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader
import torchvision.models as models
import segmentation_models_pytorch as smp

Analyse of the dataset

In [2]:
label_train = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train.csv', sep=';')
label_test = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test.csv', sep=';')


In [3]:

nouvel_ordre = ['Number', 'Disease', 'IC', 'Blur', 'LC']
mapping = {'A': 0, 'D': 1, 'G': 2, 'N': 3}
reverse = {0: 'A', 1: 'D', 2: 'G', 3: 'N'}

label_train = label_train[nouvel_ordre]
label_train['Disease'] = label_train['Disease'].map(mapping)
# L'argument index=False évite qu'un nouvel index grisé soit créé 
label_train.to_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train_clean.csv', index=False, sep=';')

label_test = label_test[nouvel_ordre]
label_test['Disease'] = label_test['Disease'].map(mapping)
label_test.to_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test_clean.csv', index=False, sep=';')

In [4]:
label_train = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/train/Quality_Assessment_train_clean.csv', sep=';')
label_test = pd.read_csv('/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset/test/Quality_Assessment_test_clean.csv', sep=';')




In [5]:
train_size = len(label_train)
test_size = len(label_test)
print("Repartition % of diseases in train and test set :")
print(label_train['Disease'].value_counts() *100 / train_size )
print(label_test['Disease'].value_counts() *100 / test_size )




Repartition % of diseases in train and test set :
Disease
0    25.0
1    25.0
2    25.0
3    25.0
Name: count, dtype: float64
Disease
0    25.0
1    25.0
2    25.0
3    25.0
Name: count, dtype: float64


In [6]:
print("Repartition % of Illumination Color in train and test set :")
print(label_train['IC'].value_counts() *100 / train_size )
print(label_test['IC'].value_counts() *100 / test_size )


print("Repartition % of Blur in train and test set :")
print(label_train['Blur'].value_counts() *100 / train_size )
print(label_test['Blur'].value_counts() *100 / test_size )


print("Repartition % of Low Contrast in train and test set :")
print(label_train['LC'].value_counts() *100 / train_size )
print(label_test['LC'].value_counts() *100 / test_size )


Repartition % of Illumination Color in train and test set :
IC
1    82.666667
0    17.333333
Name: count, dtype: float64
IC
1    74.0
0    26.0
Name: count, dtype: float64
Repartition % of Blur in train and test set :
Blur
1    85.0
0    15.0
Name: count, dtype: float64
Blur
1    79.5
0    20.5
Name: count, dtype: float64
Repartition % of Low Contrast in train and test set :
LC
1    96.666667
0     3.333333
Name: count, dtype: float64
LC
1    92.5
0     7.5
Name: count, dtype: float64


In [7]:
class CustomImageDataset(Dataset):

    def __init__(self, annotations_file, img_dir, mask_dir, img_transform=None, mask_transform=None):
       
        self.img_labels = annotations_file
        self.img_dir = img_dir # Directory where the images are stored
        self.mask_dir = mask_dir # Directory where the masks are stored
        self.img_transform = img_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.img_labels)
   
    def __getitem__(self, idx):
        
        name_image = str(self.img_labels.iloc[idx, 0]) + '_' + reverse[int(self.img_labels.iloc[idx, 1])]+ '.png'
        
        img_path = os.path.join(self.img_dir, name_image)
        mask_path = os.path.join(self.mask_dir, name_image)
        
        image = Image.open(img_path).convert('RGB') #some images have 4 channels 
        mask = Image.open(mask_path).convert('L')
        label = torch.tensor(self.img_labels.iloc[idx, 1:].values.astype('float32'), dtype=torch.float32)
        
   
        if self.img_transform:
            image = self.img_transform(image)
   
        if self.mask_transform:
            mask = self.mask_transform(mask)
   
        return image, mask, label
    

In [8]:

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor() 
])

path_dataset ="/Users/laurageneaulibourne/Downloads/S4/computer_vision/exam/datasets/dataset_retina/Fundus_Image_Vessel_dataset"
fundus_dataset_train = CustomImageDataset(annotations_file= label_train, img_dir = path_dataset + "/train/imgs", mask_dir = path_dataset + "/train/masks", img_transform=transform, mask_transform=transform)
fundus_dataset_test = CustomImageDataset(annotations_file= label_test, img_dir = path_dataset + "/test/imgs", mask_dir = path_dataset + "/test/masks", img_transform=transform, mask_transform=transform)

train_dataloader = DataLoader(fundus_dataset_train, batch_size=64, shuffle=True)
test_dataloader= DataLoader(fundus_dataset_test, batch_size=64, shuffle=True)

In [9]:
def train_loop(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()
    
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        for image, mask, label in dataloader:
        
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)
            
            optimizer.zero_grad()
        
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            
            for i in range (len(label)):
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class[i].argmax(dim=0).item() :
                   dic_acc_disease[label_name + "_true"] += 1
             
                loss_classi = loss_class(pred_class[i].unsqueeze(0), label[i,0].unsqueeze(0).long()) 
                loss_segi = loss_seg(pred_segm[i].unsqueeze(0), mask[i].unsqueeze(0))  # unsqueeze
                if ((label[i,3].item() == 0) or (label[i,2].item() == 0)): # we want to be able to detecte cataract disease index 3 for LC and index 2 for Blur
                    batch_loss_seg +=  gama * loss_segi
                    batch_loss_class += gama * loss_classi   
                else : 
                    batch_loss_seg +=  loss_segi
                    batch_loss_class += loss_classi
                    
            
            
            
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            epoch_loss_seg += batch_loss_seg.item()
            batch_loss = (batch_loss_class + batch_loss_seg) / len(label) 
            
            # Backpropagation
            batch_loss.backward()
            optimizer.step()
        
    
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg
        
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')




    

In [10]:
def test_loop(dataloader, model, loss_seg, loss_class, device, alpha, beta, gama):
    size = len(dataloader.dataset)
    
    model = model.to(device)
    model.eval()
    test_loss = 0.0
    test_loss_seg = 0.0
    test_loss_class = 0.0
    dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
    
    with torch.no_grad():
        for image, mask, label in dataloader:
            image = image.to(device)
            mask = mask.to(device)
            label = label.to(device)

            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            
            for i in range (len(label)):
                label_name = reverse[label[i,0].item()]
                dic_acc_disease[label_name + "_tot"] += 1 
                if label[i,0].item() == pred_class[i].argmax(dim=0).item() :
                   dic_acc_disease[label_name + "_true"] += 1
             
                loss_classi = loss_class(pred_class[i].unsqueeze(0), label[i,0].unsqueeze(0).long())   # unsqueeze
                loss_segi = loss_seg(pred_segm[i].unsqueeze(0), mask[i].unsqueeze(0))
                if ((label[i,3].item() == 0) or (label[i,2].item() == 0)): # we want to be able to detecte cataract disease index 3 for LC and index 2 for Blur
                    batch_loss_seg +=  gama * loss_segi
                    batch_loss_class += gama * loss_classi   
                else : 
                    batch_loss_seg +=  loss_segi
                    batch_loss_class += loss_classi
                    
            test_loss_class += alpha * batch_loss_class.item()        
            test_loss_seg += beta * batch_loss_seg.item()
             
                   

        test_loss_seg /= size
        test_loss_class /= size
        test_loss = test_loss_class + test_loss_seg
            
    print(f'Total Loss: {test_loss},Classification Loss: {test_loss_class}, Segmentation Loss: {test_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')


try to not see image by image in a batch in order to gain time 

In [24]:
def train_loop_faster(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()
    
    
    for epoch in range(num_epochs):
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        for image, mask, label in dataloader:
        
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)

            optimizer.zero_grad()
        
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
             
            loss_c= loss_class(pred_class, label[:,0].long()) 
            loss_s = loss_seg(pred_segm, mask)  
            
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))
           
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = (loss_s.mean(dim=[1, 2, 3]) * weights).sum()  #######
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            batch_loss = (batch_loss_class + batch_loss_seg ) / len(label)
            epoch_loss_seg += batch_loss_seg.item()
            
            
            pred_label = pred_class.argmax(dim=1)
            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                

            # Backpropagation
            batch_loss.backward()
            optimizer.step()

        
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')



In [12]:
def test_loop_faster(dataloader, model, loss_seg, loss_class, device, alpha, beta, gama):
    size = len(dataloader.dataset)
    
    model = model.to(device)
    model.eval()
    test_loss = 0.0
    test_loss_seg = 0.0
    test_loss_class = 0.0
    dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
    
    with torch.no_grad():
        for image, mask, label in dataloader:
            image = image.to(device)
            mask = mask.to(device)
            label = label.to(device)

            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
            
            loss_c= loss_class(pred_class, label[:,0].long()) 
            loss_s = loss_seg(pred_segm, mask)  # unsqueeze
            
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))
           
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = (loss_s.mean(dim=[1, 2, 3]) * weights).sum()  #######
            batch_loss_class = alpha * batch_loss_class
            batch_loss_seg = beta * batch_loss_seg
            
            test_loss_seg += batch_loss_seg.item()
            test_loss_class += batch_loss_class.item()
            
            
            pred_label = pred_class.argmax(dim=1)
            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                
            
        test_loss_seg /= size
        test_loss_class /= size  
        test_loss = test_loss_class + test_loss_seg
        
            
    print(f'Total Loss: {test_loss},Classification Loss: {test_loss_class}, Segmentation Loss: {test_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')


In [31]:
'''
def train_loop_faster2(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):

    size = len(dataloader.dataset)  
    model = model.to(device)
    model.train()
    
    
    for epoch in range(num_epochs):
        epoch_loss_class = 0.0
        epoch_loss_seg = 0.0
        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}
        for image, mask, label in dataloader:
        
            image= image.to(device)
            mask= mask.to(device)
            label= label.to(device)
            
            optimizer.zero_grad()
        
            pred_segm, pred_class = model(image)
            
            batch_loss_class = 0
            batch_loss_seg = 0
             
            loss_c= loss_class(pred_class, label[:,0].long()) 
            loss_s = loss_seg(pred_segm, mask)  # unsqueeze
            
            penalty = (label[:, 2] == 0) | (label[:, 3] == 0)
            
            weights = torch.where(penalty, torch.tensor(gama, dtype=torch.float32, device=device), torch.tensor(1.0, dtype=torch.float32, device=device))
            weights_seg = weights.view(-1, 1, 1, 1)
            
            batch_loss_class = (loss_c * weights).sum()
            batch_loss_seg = loss_seg(pred_segm * weights_seg, mask * weights_seg) * image.size(0)
            batch_loss_class = alpha * batch_loss_class
            epoch_loss_class += batch_loss_class.item()
            batch_loss_seg = beta * batch_loss_seg
            raw_loss_seg = loss_seg(pred_segm, mask) 
            batch_loss_seg = raw_loss_seg * weights.mean() * image.size(0)
            batch_loss = batch_loss_class + batch_loss_seg
            epoch_loss_seg += batch_loss_seg.item()
            
            
            pred_label = pred_class.argmax(dim=1)
            for i in range (len(dic_acc_disease) // 2):
                disease = reverse[i]
                idx_pred = (pred_label == i)
                idx_true = (label[:, 0] == i)
                
                dic_acc_disease[disease + "_tot"] += idx_true.sum().item()
                dic_acc_disease[disease + "_true"] += (idx_pred & idx_true).sum().item()
                

            # Backpropagation
            batch_loss.backward()
            optimizer.step()
            
        
        epoch_loss_class /= size
        epoch_loss_seg /= size
        epoch_loss = epoch_loss_class + epoch_loss_seg
        
        print(f'Epoch {epoch+1}/{num_epochs}, Total Loss: {epoch_loss},Classification Loss: {epoch_loss_class}, Segmentation Loss: {epoch_loss_seg}, Accuracy diseases: A : {dic_acc_disease["A_true"] *100 / dic_acc_disease["A_tot"]} %, D :{dic_acc_disease["D_true"] *100 / dic_acc_disease["D_tot"]} %, G :{dic_acc_disease["G_true"] *100 / dic_acc_disease["G_tot"]} %, N :{dic_acc_disease["N_true"] *100 / dic_acc_disease["N_tot"]} %)')


'''

'\ndef train_loop_faster2(dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama):\n\n    size = len(dataloader.dataset)  \n    model = model.to(device)\n    model.train()\n    \n    \n    for epoch in range(num_epochs):\n        epoch_loss_class = 0.0\n        epoch_loss_seg = 0.0\n        dic_acc_disease = {"A_true": 0,"A_tot": 0,  "D_true": 0, "D_tot" : 0,  "G_true": 0, "G_tot" : 0, "N_true": 0, "N_tot": 0}\n        for image, mask, label in dataloader:\n        \n            image= image.to(device)\n            mask= mask.to(device)\n            label= label.to(device)\n            \n            optimizer.zero_grad()\n        \n            pred_segm, pred_class = model(image)\n            \n            batch_loss_class = 0\n            batch_loss_seg = 0\n             \n            loss_c= loss_class(pred_class, label[:,0].long()) \n            loss_s = loss_seg(pred_segm, mask)  # unsqueeze\n            \n            penalty = (label[:, 2] == 0) 

In [22]:

# segmentation :
model_segmentation = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",     
    in_channels=3,                 
    classes=1,                      
)

# global classification  :
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    aux_params=dict(
        pooling='avg',             
        dropout=0.2,               
        classes=4,               
    )
)



In [ ]:
model_segmentation2 = smp.Unet(
    encoder_name="resnet18",        
    encoder_weights="imagenet",     
    in_channels=3,                 
    classes=1,                      
)

# global classification  :
model2 = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    aux_params=dict(
        pooling='avg',             
        dropout=0.2,               
        classes=4, # A,d, G, N           
    )
)

In [ ]:
loss_seg= torch.nn.BCEWithLogitsLoss()  #############
loss_class = nn.CrossEntropyLoss()
loss_seg_faster2= torch.nn.BCEWithLogitsLoss(reduction='mean')  ###########
loss_seg_faster= torch.nn.BCEWithLogitsLoss(reduction='none')
loss_class_faster = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
alpha = 1
beta = 1
gama = 2


In [34]:
train_loop(train_dataloader, model, loss_seg, loss_class, optimizer, num_epochs, device, alpha, beta, gama)

Epoch 1/10, Total Loss: 2.5284521373113,Classification Loss: 1.8527982807159424, Segmentation Loss: 0.6756538565953573, Accuracy diseases: A : 49.333333333333336 %, D :46.0 %, G :60.0 %, N :74.66666666666667 %)
Epoch 2/10, Total Loss: 1.1090078496932985,Classification Loss: 0.7288206768035889, Segmentation Loss: 0.3801871728897095, Accuracy diseases: A : 70.66666666666667 %, D :54.666666666666664 %, G :80.66666666666667 %, N :94.66666666666667 %)
Epoch 3/10, Total Loss: 0.8318246547381083,Classification Loss: 0.5447526454925538, Segmentation Loss: 0.2870720092455546, Accuracy diseases: A : 73.33333333333333 %, D :68.0 %, G :88.66666666666667 %, N :96.0 %)
Epoch 4/10, Total Loss: 0.6486023004849751,Classification Loss: 0.4081174564361572, Segmentation Loss: 0.24048484404881795, Accuracy diseases: A : 86.0 %, D :74.0 %, G :88.0 %, N :99.33333333333333 %)
Epoch 5/10, Total Loss: 0.6167197132110596,Classification Loss: 0.40340613762537636, Segmentation Loss: 0.21331357558568317, Accuracy d

In [ ]:
train_loop(train_dataloader, model2, loss_seg, loss_class, optimizer2, num_epochs, device, alpha, beta, gama)

In [ ]:
test_loop(test_dataloader, model, loss_seg, loss_class, device, alpha, beta, gama)

Total Loss: 1.882282260656357,Classification Loss: 1.7121078729629517, Segmentation Loss: 0.17017438769340515, Accuracy diseases: A : 84.0 %, D :56.0 %, G :92.0 %, N :22.0 %)


In [ ]:
test_loop(test_dataloader, model2, loss_seg, loss_class, device, alpha, beta, gama)

In [23]:
train_loop_faster(train_dataloader, model, loss_seg_faster, loss_class_faster, optimizer, num_epochs, device, alpha, beta, gama)

tensor([1., 0., 1., 2., 1., 0., 2., 3., 0., 1., 0., 0., 2., 0., 0., 1., 0., 3.,
        3., 2., 2., 3., 3., 2., 1., 3., 2., 0., 1., 3., 0., 0., 1., 3., 0., 0.,
        0., 1., 2., 3., 3., 0., 3., 0., 3., 1., 1., 1., 2., 0., 3., 1., 1., 3.,
        0., 3., 3., 0., 2., 0., 2., 3., 1., 1.])
tensor([0., 1., 2., 3.])


ZeroDivisionError: division by zero

In [56]:
test_loop_faster(test_dataloader, model, loss_seg_faster, loss_class_faster, device, alpha, beta, gama)

Total Loss: 2.823963644504547,Classification Loss: 1.9471197938919067, Segmentation Loss: 0.8768438506126404, Accuracy diseases: A : 2.0 %, D :28.0 %, G :44.0 %, N :6.0 %)
